#  Geolocation - Bronze Ingestion

## Imports


In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, DecimalType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_geolocation"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist"
source_dataset = "geolocation"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("geolocation_zip_code_prefix", StringType(), True),
    StructField("geolocation_lat", DecimalType(16, 14), True),
    StructField("geolocation_lng", DecimalType(16, 14), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: decimal(16,14) (nullable = true)
 |-- geolocation_lng: decimal(16,14) (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-02T22:38:49.325Z,4f513a96-794e-4af1-99de-a04e94216699,olist,geolocation
01046,-23.54608112703554,-46.64482029837157,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-02T22:38:49.325Z,4f513a96-794e-4af1-99de-a04e94216699,olist,geolocation
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-02T22:38:49.325Z,4f513a96-794e-4af1-99de-a04e94216699,olist,geolocation
01041,-23.54439216486810,-46.63949930627844,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-02T22:38:49.325Z,4f513a96-794e-4af1-99de-a04e94216699,olist,geolocation
01035,-23.54157796171149,-46.64160722329613,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-02T22:38:49.325Z,4f513a96-794e-4af1-99de-a04e94216699,olist,geolocation


In [0]:
spark.table(target_table).count()

1000163